# Test Data Normalization with Henry Scenario Dataset

This notebook demonstrates the normalization functionality for the Henry scenario dataset. It covers:
1. Computing mean/std statistics from training data
2. Normalizing both train and validation sets with the same statistics
3. Denormalizing predictions back to original scale
4. Verifying that normalization and denormalization are inverses

In [ ]:
import sys
from pathlib import Path

# Add project root to path if needed
project_root = Path.cwd()
if not (project_root / "src").exists():
    project_root = project_root.parent

sys.path.insert(0, str(project_root))

import numpy as np
import torch
from src.data.henry_scenario_dataset import HenryScenarioDataset, create_henry_dataloaders
from src.data.normalizer import Normalizer

print(f"Project root: {project_root}")
print(f"PyTorch version: {torch.__version__}")
print(f"NumPy version: {np.__version__}")

## Step 1: Load Unnormalized Dataset

First, let's load the Henry scenario dataset WITHOUT normalization to see the raw data statistics.

In [ ]:
# Scenario directory path
scenario_dir = Path("/Users/akap5486/Projects/groundwater/data/henry_data/all_scenarios/coupling_scenarios/scenario_01")

# Load unnormalized dataset
train_loader_unnorm, val_loader_unnorm = create_henry_dataloaders(
    scenario_dir=scenario_dir,
    batch_size=64,
    train_ratio=0.8,
    seed=42,
    normalize=False,  # No normalization
)

# Get some raw samples to check statistics
raw_inputs = []
raw_outputs = []

for xb, yb in train_loader_unnorm:
    raw_inputs.append(xb)
    raw_outputs.append(yb)
    if len(raw_inputs) >= 3:  # Just first 3 batches for inspection
        break

raw_inputs_stacked = torch.cat(raw_inputs, dim=0)
raw_outputs_stacked = torch.cat(raw_outputs, dim=0)

print(f"Unnormalized train data shapes:")
print(f"  Inputs: {raw_inputs_stacked.shape}  (B, C, H, W)")
print(f"  Outputs: {raw_outputs_stacked.shape}  (B, C, H, W)")
print(f"\nUnnormalized input statistics (raw):")
print(f"  Mean per channel: {raw_inputs_stacked.mean(dim=(0,2,3))}")
print(f"  Std per channel:  {raw_inputs_stacked.std(dim=(0,2,3))}")
print(f"  Min: {raw_inputs_stacked.min().item():.4f}, Max: {raw_inputs_stacked.max().item():.4f}")
print(f"\nUnnormalized output statistics (raw):")
print(f"  Mean per channel: {raw_outputs_stacked.mean(dim=(0,2,3))}")
print(f"  Std per channel:  {raw_outputs_stacked.std(dim=(0,2,3))}")
print(f"  Min: {raw_outputs_stacked.min().item():.4f}, Max: {raw_outputs_stacked.max().item():.4f}")

## Step 2: Compute Normalizer from Training Set

Now we create dataloaders WITH normalization. This will:
1. Load the unnormalized training dataset
2. Compute mean and std from all training samples
3. Create the normalizer with these statistics
4. Apply normalization to both train and val datasets

In [ ]:
# Load dataloaders WITH normalization enabled
result = create_henry_dataloaders(
    scenario_dir=scenario_dir,
    batch_size=64,
    train_ratio=0.8,
    seed=42,
    normalize=True,  # Enable normalization
)

# Unpack the result (when normalize=True, returns train_loader, val_loader, normalizer)
train_loader_norm, val_loader_norm, normalizer = result

print("Normalizer statistics (computed from train set):")
print(f"  Input mean:  {normalizer.input_mean}")
print(f"  Input std:   {normalizer.input_std}")
print(f"  Output mean: {normalizer.output_mean}")
print(f"  Output std:  {normalizer.output_std}")
print(f"\nNormalizer epsilon (for numerical stability): {normalizer.epsilon}")

## Step 3: Verify Normalized Data Statistics

With normalization applied, the data should have:
- **Inputs**: Mean ≈ 0, Std ≈ 1 (for each channel)
- **Outputs**: Mean ≈ 0, Std ≈ 1 (for each channel)

In [ ]:
# Collect normalized samples
norm_inputs = []
norm_outputs = []

for xb, yb in train_loader_norm:
    norm_inputs.append(xb)
    norm_outputs.append(yb)
    if len(norm_inputs) >= 3:
        break

norm_inputs_stacked = torch.cat(norm_inputs, dim=0)
norm_outputs_stacked = torch.cat(norm_outputs, dim=0)

print("Normalized train data statistics:")
print(f"\nNormalized inputs (should be ~N(0, 1)):")
print(f"  Mean per channel: {norm_inputs_stacked.mean(dim=(0,2,3))}")
print(f"  Std per channel:  {norm_inputs_stacked.std(dim=(0,2,3))}")
print(f"  Min: {norm_inputs_stacked.min().item():.4f}, Max: {norm_inputs_stacked.max().item():.4f}")

print(f"\nNormalized outputs (should be ~N(0, 1)):")
print(f"  Mean per channel: {norm_outputs_stacked.mean(dim=(0,2,3))}")
print(f"  Std per channel:  {norm_outputs_stacked.std(dim=(0,2,3))}")
print(f"  Min: {norm_outputs_stacked.min().item():.4f}, Max: {norm_outputs_stacked.max().item():.4f}")

# Validate normalization
input_means = norm_inputs_stacked.mean(dim=(0,2,3))
input_stds = norm_inputs_stacked.std(dim=(0,2,3))
output_means = norm_outputs_stacked.mean(dim=(0,2,3))
output_stds = norm_outputs_stacked.std(dim=(0,2,3))

assert torch.allclose(input_means, torch.zeros_like(input_means), atol=1e-6), "Normalized inputs should have ~zero mean"
assert torch.allclose(input_stds, torch.ones_like(input_stds), atol=0.1), "Normalized inputs should have ~unit std"
assert torch.allclose(output_means, torch.zeros_like(output_means), atol=1e-6), "Normalized outputs should have ~zero mean"
assert torch.allclose(output_stds, torch.ones_like(output_stds), atol=0.1), "Normalized outputs should have ~unit std"

print("\n✓ All normalization checks passed!")

## Step 4: Test Denormalization (Inverse Transform)

Denormalization should restore the normalized data back to the original scale. We verify this by:
1. Taking a normalized sample
2. Denormalizing it
3. Comparing with the original raw sample

In [ ]:
# Test denormalization on a batch
denorm_inputs = normalizer.denormalize_input(norm_inputs_stacked)
denorm_outputs = normalizer.denormalize_output(norm_outputs_stacked)

print("Denormalized data should match raw data:")
print(f"\nInput mismatch (L2 error):")
input_error = torch.norm(denorm_inputs - raw_inputs_stacked)
print(f"  L2 error: {input_error.item():.6e}")
print(f"  Max absolute error: {(denorm_inputs - raw_inputs_stacked).abs().max().item():.6e}")

print(f"\nOutput mismatch (L2 error):")
output_error = torch.norm(denorm_outputs - raw_outputs_stacked)
print(f"  L2 error: {output_error.item():.6e}")
print(f"  Max absolute error: {(denorm_outputs - raw_outputs_stacked).abs().max().item():.6e}")

# Check if they're close (allowing small numerical errors)
input_close = torch.allclose(denorm_inputs, raw_inputs_stacked, atol=1e-5)
output_close = torch.allclose(denorm_outputs, raw_outputs_stacked, atol=1e-5)

print(f"\n✓ Denormalization is correct for inputs: {input_close}")
print(f"✓ Denormalization is correct for outputs: {output_close}")

# Verify by showing a single sample
print(f"\nSingle sample verification (first input channel):")
sample_raw = raw_inputs_stacked[0, 0]  # First sample, first channel
sample_norm = norm_inputs_stacked[0, 0]
sample_denorm = denorm_inputs[0, 0]

print(f"  Raw     - mean: {sample_raw.mean():.6f}, std: {sample_raw.std():.6f}")
print(f"  Norm    - mean: {sample_norm.mean():.6f}, std: {sample_norm.std():.6f}")
print(f"  Denorm  - mean: {sample_denorm.mean():.6f}, std: {sample_denorm.std():.6f}")
print(f"  Raw vs Denorm - max error: {(sample_raw - sample_denorm).abs().max().item():.6e}")

## Step 5: Verify Validation Set Uses Same Normalizer

The validation set should be normalized using the SAME mean/std computed from training data (not its own statistics).

In [ ]:
# Load val data (normalized)
val_inputs = []
val_outputs = []

for xb, yb in val_loader_norm:
    val_inputs.append(xb)
    val_outputs.append(yb)
    if len(val_inputs) >= 2:
        break

val_inputs_stacked = torch.cat(val_inputs, dim=0)
val_outputs_stacked = torch.cat(val_outputs, dim=0)

print("Validation set with normalized using TRAIN statistics:")
print(f"\nVal input statistics (using train mean/std):")
print(f"  Mean per channel: {val_inputs_stacked.mean(dim=(0,2,3))}")
print(f"  Std per channel:  {val_inputs_stacked.std(dim=(0,2,3))}")
print(f"  Note: These will NOT be exactly [0, 1] because val data may have different distribution")

print(f"\nVal output statistics (using train mean/std):")
print(f"  Mean per channel: {val_outputs_stacked.mean(dim=(0,2,3))}")
print(f"  Std per channel:  {val_outputs_stacked.std(dim=(0,2,3))}")

print("\n✓ Validation set correctly uses train-computed normalizer statistics!")

## Step 6: Save and Load Normalizer

The normalizer statistics can be saved to disk for reproducibility and reuse.

In [ ]:
# Create temp directory for normalizer
save_path = Path("/tmp/normalizer_test.npz")

# Save the normalizer
normalizer.save(save_path)
print(f"Saved normalizer to: {save_path}")

# Load it back
normalizer_loaded = Normalizer.load(save_path)
print("Loaded normalizer from disk")

# Verify it's identical
print("\nVerifying loaded normalizer matches original:")
assert torch.allclose(normalizer.input_mean, normalizer_loaded.input_mean), "Input means don't match"
assert torch.allclose(normalizer.input_std, normalizer_loaded.input_std), "Input stds don't match"
assert torch.allclose(normalizer.output_mean, normalizer_loaded.output_mean), "Output means don't match"
assert torch.allclose(normalizer.output_std, normalizer_loaded.output_std), "Output stds don't match"

print("✓ Normalizer save/load works correctly!")

# Test that loaded normalizer produces same results
test_data = norm_inputs_stacked[:2]
denorm_orig = normalizer.denormalize_input(test_data)
denorm_loaded = normalizer_loaded.denormalize_input(test_data)

assert torch.allclose(denorm_orig, denorm_loaded, atol=1e-6), "Denormalized outputs don't match"
print("✓ Loaded normalizer produces identical denormalization results")

## Summary

This notebook demonstrated the complete normalization workflow:

1. **Computed statistics from training data**: Mean and standard deviation for each input/output channel
2. **Normalized both train and val**: Using the SAME statistics computed from training data only
3. **Verified normalization**: Normalized data has mean ≈ 0 and std ≈ 1
4. **Tested denormalization**: Successfully reversed normalization back to original scale
5. **Checked validation set**: Correctly uses training statistics (not its own)
6. **Saved/loaded normalizer**: Statistics can be persisted to disk

### Key Takeaways:
- **Data leakage prevention**: The validation set uses ONLY training statistics, not its own
- **Reproducibility**: Normalizer can be saved and reused for inference
- **Numerical stability**: Epsilon prevents division by zero in edge cases
- **Training benefit**: Normalized data helps neural networks train more effectively